# Stripe analysis — how striped are the EI maps vs the RGB?

**What/why:** decide whether to destripe, and where. **Approach:** no-reference *streaking metric*
(standard when no clean reference exists) — mean absolute deviation of each column mean from its
neighbours, normalised by the robust range (p99−p1, since EI is signed):

    streaking = mean_j |mu_j − (mu_{j-1}+mu_{j+1})/2| / (p99−p1)

Higher = more stripe. Section 1: how much (EI vs RGB). Section 2: constant vs value-dependent.

# Section 1 — Stripe strength: is the stripe in the EI, the RGB, or both?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

import config

PROJECT_ROOT = Path(config.__file__).resolve().parent
LOCAL        = (PROJECT_ROOT / config.LOCAL_PROCESSED_DIR).resolve()
EI_MAPS_DIR  = LOCAL / "ei_maps"
manifest     = pd.read_csv(LOCAL / "manifest.csv")

def streaking_metric(img2d):
    mu  = img2d.mean(axis=0)                                   # column means (mean over rows)
    ref = (mu[:-2] + mu[2:]) / 2                               # immediate-neighbour reference
    dev = np.abs(mu[1:-1] - ref)                               # per-column deviation from neighbours
    rng = np.percentile(img2d, 99) - np.percentile(img2d, 1)   # robust range (handles signed EI)
    return float(dev.mean() / rng)

print("EI maps present:", EI_MAPS_DIR.exists(), " | images:", len(manifest))

## Step 1 — streaking of the EI maps (all 306, local)

In [ ]:
rows = []
for r in manifest.itertuples(index=False):
    ei = np.load(EI_MAPS_DIR / f"{r.subject_id}_{r.pose}_{r.view}.npy")
    rows.append(dict(split=r.split, ei_streak=streaking_metric(ei)))
ei_df = pd.DataFrame(rows)

print("EI streaking metric:")
print(f"  ALL   mean={ei_df.ei_streak.mean():.4f}  p50={ei_df.ei_streak.median():.4f}  p95={ei_df.ei_streak.quantile(.95):.4f}")
for s in ["train", "valid", "test"]:
    g = ei_df[ei_df.split == s]
    print(f"  {s:<5} mean={g.ei_streak.mean():.4f}")

## Step 2 — streaking of the RGB images (all 306, needs the SSD)

Same metric, applied to RGB luminance (mean over the 3 channels). Directly comparable to the
EI number because both are normalised by their own robust range.

In [ ]:
from src.io_utils import load_rgb

rows = []
for r in manifest.itertuples(index=False):
    rgb = load_rgb(str(r.rgb_path))                   # (H, W, 3)
    rows.append(dict(split=r.split, rgb_streak=streaking_metric(rgb.mean(2))))  # luminance
rgb_df = pd.DataFrame(rows)

print("RGB streaking metric (luminance = mean over channels):")
print(f"  ALL   mean={rgb_df.rgb_streak.mean():.4f}  p50={rgb_df.rgb_streak.median():.4f}  p95={rgb_df.rgb_streak.quantile(.95):.4f}")
for s in ["train", "valid", "test"]:
    g = rgb_df[rgb_df.split == s]
    print(f"  {s:<5} mean={g.rgb_streak.mean():.4f}")


In [ ]:
# Step 3 — compare EI vs RGB streaking
ei_m, rgb_m = ei_df.ei_streak.mean(), rgb_df.rgb_streak.mean()
print(f"EI  streaking (mean) = {ei_m:.4f}")
print(f"RGB streaking (mean) = {rgb_m:.4f}")
print(f"EI / RGB             = {ei_m / rgb_m:.1f}x   -> the stripe lives in the EI; RGB is ~clean")


## Conclusion

| | streaking (mean) |
|---|---|
| EI maps | **0.077** |
| RGB | **0.005** (~16× less) |

The stripe is almost entirely in the EI (`log(1/R)` amplifies the per-column sensor offset); RGB is
near-clean and uncorrelated with it. **Decision: destripe the EI target, leave RGB raw.**

# Section 2 — constant offset or value-dependent?

**Why:** an additive stripe (`measured = true + offset_col`) is fixed by subtracting one offset per
column (post-hoc); a value-dependent one (`gain_col·true + offset_col`) is not — it needs the
cube/reflectance level before the `log`. **Approach:** build a stripe-free reference (horizontal
median), measure each column's offset over its dark vs bright pixels, regress bright-on-dark across
columns — slope ≈1 = constant, slope <1 = value-dependent.

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

def dark_bright_offsets(ei, refwin=51, min_px=30):
    """Per-column stripe offset measured separately in dark vs bright pixels."""
    ref   = median_filter(ei, size=(1, refwin))      # horizontal median -> stripe-free reference
    resid = ei - ref                                 # per-pixel stripe estimate
    low, high = ref < np.median(ref), ref >= np.median(ref)
    nl, nh = low.sum(0), high.sum(0)
    off_low  = np.where(nl > min_px, (resid * low ).sum(0) / np.maximum(nl, 1), np.nan)
    off_high = np.where(nh > min_px, (resid * high).sum(0) / np.maximum(nh, 1), np.nan)
    return off_low, off_high

# Aggregate: slope of (bright vs dark) offset per image, across all 306
slopes = []
for r in manifest.itertuples(index=False):
    ei = np.load(EI_MAPS_DIR / f"{r.subject_id}_{r.pose}_{r.view}.npy")
    ol, oh = dark_bright_offsets(ei)
    m = np.isfinite(ol) & np.isfinite(oh)
    if m.sum() > 50:
        slopes.append(np.polyfit(ol[m], oh[m], 1)[0])
slopes = np.array(slopes)
print(f"bright-vs-dark stripe slope over {len(slopes)} images: "
      f"mean={slopes.mean():.2f}  p50={np.median(slopes):.2f}  p05={np.percentile(slopes,5):.2f}  p95={np.percentile(slopes,95):.2f}")
print(f"  value-dependent (slope < 0.9): {100*np.mean(slopes < 0.9):.0f}% of images")
print(f"  ~constant       (0.9-1.1)    : {100*np.mean((slopes>=0.9)&(slopes<=1.1)):.0f}% of images")

# Visual: dark vs bright offset per column for the 3 disclosure subjects
DISCLOSURE_SUBJECTS = ["p012", "p019", "p027"]
fig, axes = plt.subplots(1, len(DISCLOSURE_SUBJECTS), figsize=(15, 4.5))
for ax, s in zip(axes, DISCLOSURE_SUBJECTS):
    ei = np.load(EI_MAPS_DIR / f"{s}_neutral_left.npy")
    ol, oh = dark_bright_offsets(ei); m = np.isfinite(ol) & np.isfinite(oh)
    sl = np.polyfit(ol[m], oh[m], 1)[0]
    lim = max(np.abs(ol[m]).max(), np.abs(oh[m]).max())
    ax.scatter(ol[m], oh[m], s=6, alpha=0.4)
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1, label="y=x (constant)")
    ax.plot([-lim, lim], [-lim*sl, lim*sl], "r-", lw=1, label=f"fit slope={sl:.2f}")
    ax.set_xlabel("stripe offset — DARK pixels"); ax.set_ylabel("stripe offset — BRIGHT pixels")
    ax.set_title(s); ax.legend(fontsize=8)
plt.suptitle("Per-column stripe: dark vs bright offset  (on y=x -> constant;  below -> value-dependent)", y=1.02)
plt.tight_layout(); plt.show()


## Section 2 — Finding

Bright-vs-dark offset slope: **mean ≈ 0.71, 97% of images < 0.9** → the stripe is **value-dependent**
dataset-wide (~30% weaker on skin than background), matching the additive-error-through-`log` model.

**Consequences:** (1) post-hoc per-column destriping removes only the average — a value-dependent
residual remains (cube-level would be needed to fully remove it); (2) but the stripe is weakest on
skin and masking drops the background, so the residual on the masked target is milder than 0.077 implies.